# 05 · Trace e log

Le relazioni dentro una run: l'albero di una richiesta, il grafo delle chiamate tra servizi, latenza ed errori per servizio nel tempo, dove nascono gli errori, i log, e tutto insieme alle metriche.

Le tabelle vengono da `anomaly/typed.py`: una riga per span, una riga per riga di log. Le trace vengono esportate tutte, ma il campionamento si decide alla radice: Hotel traccia il 100% delle richieste, Social il 60% (il 100% in pilot2; l'1% in pilot e sweep, dove l'export ne teneva al massimo 5 al secondo e `trace_sampling` manca). `requests` conta il campione; `rps` lo riporta al traffico dividendo per `trace_sampling`, ed è NaN dove il campionamento non è noto. V5 dice se le trace della run sono integre: Jaeger non riavviato, radici pari alle richieste consegnate da wrk2, nessun blocco dell'export pieno sotto carico. I log non contengono l'id della trace né dello span: si uniscono a trace e metriche solo per tempo e pod in Hotel, per tempo e servizio in Social, dove lo span porta il nome del servizio al posto del pod (ogni servizio ha una replica sola, quindi è lo stesso).

In [ ]:
import sys
import warnings
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "anomaly" / "build.py").exists())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from anomaly import explore as ex

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_colwidth", 90)
from anomaly import build, typed

In [ ]:
DATA = Path("/home/tecnico/datasets/pilot")  # cartella del dataset: pilot, pilot2, sweep, trials, o una loro sottocartella (pilot/hotel)
RUN = 26      # quale run: None = prima run di test con un guasto; un numero della lista (7); un pezzo del nome ("network_delay")
STEP = 30       # passo della griglia, lo stesso delle metriche
TRACE = None    # id di una trace; None prende la prima che fallisce dopo l'iniezione, altrimenti la più lunga
SERVICE = None  # servizio del confronto finale; None prende il target, altrimenti l'entry point

runs = ex.find_runs(DATA)
if not runs:
    raise FileNotFoundError(f"Nessuna run valida in {DATA}: controlla DATA")
for number, run_id in enumerate(runs):
    print(f"{number:3d}  {run_id}")

In [ ]:
if RUN is None:
    run_id = next((r for r in runs if "test" in r.split("/") and "noop" not in r), next(iter(runs)))
elif isinstance(RUN, int):
    run_id = list(runs)[RUN]
else:
    matches = [RUN] if RUN in runs else [r for r in runs if RUN in r]
    if len(matches) != 1:
        raise ValueError(f"RUN={RUN!r} corrisponde a {len(matches)} run, ne serve una: {matches}")
    run_id = matches[0]
run = ex.load(runs[run_id], STEP)
L = run.labels
spans = typed.add_callees(typed.spans(run.attempt), ex.DATAGRAPH_DIR / build.DATAGRAPH[L["app_scenario"]])
logs = typed.logs(run.attempt)
inject = L["t_inject"]
minutes = run.t_rel / 60
print(f"{run_id}: {L['fault_type']}, target {L['target']}")
print(f"  {spans.trace.nunique()} trace, {len(spans)} span, {spans.service.nunique()} servizi, {int(spans.error.sum())} span in errore")
print(f"  {len(logs)} righe di log:", logs.level.replace("", "senza livello").value_counts().to_dict())
print("  V5 trace integre:", *typed.trace_check(spans, run.attempt, L))

## Una trace

Ogni riga è uno span, sotto il suo padre: servizio, tipo (server riceve, client chiama), durata, operazione, e per gli errori codice e messaggio. Un client senza figli è una chiamata che nessuno ha ricevuto: un database (non tracciato) oppure un servizio che non risponde.

In [ ]:
failing = typed.error_origins(spans)
failing_after = failing[failing.start_us / 1e6 > inject]
trace_id = TRACE or (failing_after.trace.iloc[0] if len(failing_after) else spans.groupby("trace").size().idxmax())
print(f"trace {trace_id}")
print("\n".join(typed.trace_tree(spans, trace_id)))

## Grafo delle chiamate contro il datagraph

Una chiamata è uno span figlio in un servizio diverso da quello del padre. `unanswered` = chiamate senza alcun figlio, mostrate verso il nodo chiamato (o come `? operazione` quando non si risolve): per database e cache è normale, per un servizio è un guasto. L'ultima colonna dice se l'arco esiste tra i `CALLS` o gli `USES` del datagraph usato dagli agenti.

In [ ]:
edges = typed.call_edges(spans)
graph = build.datagraph_edges(ex.DATAGRAPH_DIR / build.DATAGRAPH[L["app_scenario"]])
calls = {(a, b) for a, relation, b in graph if relation == "CALLS"}
links = {(a, b) for a, _, b in graph}
edges["nel datagraph"] = ["-" if b.startswith("?") else (a, b) in links for a, b in zip(edges.caller, edges.callee)]
print("Archi CALLS del datagraph mai visti nelle trace:", sorted(calls - set(zip(edges.caller, edges.callee))) or "nessuno")
edges

## Latenza ed errori per servizio nel tempo

Sulla stessa griglia delle metriche: p95 delle richieste servite e quota di quelle in errore, per servizio; la p95 è vuota dove le richieste del passo sono meno di `typed.MIN_SPANS`. La linea nera è l'iniezione.

In [ ]:
feats = typed.span_features(spans, run.t, STEP, L["trace_sampling"])
services = [s for s in run.services if s in set(feats.service[feats.requests.notna()])]  # i nodi che servono richieste
extent = [minutes[0], minutes[-1], len(services) - 0.5, -0.5]
fig, axes = plt.subplots(1, 2, figsize=(18, 0.25 * len(services) + 2), sharey=True, constrained_layout=True)
for a, (column, title, cmap) in zip(axes, [("p95_ms", "latenza p95 (ms)", "viridis"), ("error_share", "quota di richieste in errore", "Reds")]):
    grid = feats.pivot(index="service", columns="t", values=column).reindex(index=services, columns=run.t).to_numpy(float)
    vmax = 1 if column == "error_share" else np.nanpercentile(grid, 99) if np.isfinite(grid).any() else 1
    image = a.imshow(grid, aspect="auto", cmap=cmap, vmin=0, vmax=vmax, extent=extent, interpolation="nearest")
    a.set_title(title)
    a.axvline(0, color="k")
    a.set_yticks(range(len(services)), services, fontsize=7)
    a.set_xlabel("minuti dall'iniezione")
    fig.colorbar(image, ax=a, shrink=0.7)
plt.show()

## Dove nascono gli errori

Sopra: le chiamate senza risposta verso servizi che crescono di più dopo l'iniezione rispetto al settle (quelle verso database e cache sono normali e restano fuori), con il nodo che non ha risposto. Sotto: l'origine degli errori, cioè lo span in errore più profondo di ogni trace, che non ha figli in errore. Hotel marca gli errori in ogni servizio; Social solo in nginx, quindi lì l'origine cade su nginx. In entrambe il nodo rotto spesso non ha span propri (un servizio caduto): lo indica il chiamato delle chiamate senza risposta o fallite, che `span_features` conta come `unanswered_share` e `failed_share`.

In [ ]:
lonely = typed.unanswered_calls(spans)
lonely = lonely[~lonely.datastore]
lonely = lonely.assign(minute=(lonely.start_us / 1e6 - inject) / 60,
                       call=lonely.service + " → " + lonely.callee.where(lonely.callee != "", "? " + lonely.operation))
before = lonely[(lonely.minute > minutes[0]) & (lonely.minute <= 0)].call.value_counts() / max(-minutes[0], 1)
after = lonely[lonely.minute > 0].call.value_counts() / max(minutes[-1], 1)
growth = (after - before.reindex(after.index).fillna(0)).sort_values(ascending=False)
bins = np.arange(minutes[0], minutes[-1] + STEP / 60, STEP / 60)
fig, a = plt.subplots(figsize=(13, 3.5))
for call in growth.head(6).index:
    a.hist(lonely[lonely.call == call].minute, bins=bins, histtype="step", label=call)
a.axvline(0, color="red", ls="--")
a.set(xlabel="minuti dall'iniezione", ylabel="chiamate senza risposta per passo", title="Chiamate senza risposta che crescono dopo l'iniezione")
if len(growth):
    a.legend(fontsize=7)
plt.show()

if len(failing_after):
    display(failing_after.groupby(["service", "operation", "status"]).size().sort_values(ascending=False).head(10).rename("trace"))
else:
    print("Nessuno span in errore dopo l'iniezione")

## Log

Righe di livello warning, error o fatal per servizio e passo; sotto, i messaggi più frequenti dopo l'iniezione, con i numeri sostituiti da `#` così che righe uguali a meno di un id si contino insieme.

In [ ]:
bad = logs[logs.level.isin(["warning", "error", "fatal"]) & (logs.t > run.t[0] - STEP) & (logs.t <= run.t[-1])]
if len(bad):
    cell = run.t[np.minimum(np.searchsorted(run.t, bad.t), len(run.t) - 1)]
    table = pd.crosstab(bad.service, cell).reindex(columns=run.t, fill_value=0)
    fig, a = plt.subplots(figsize=(13, 0.3 * len(table) + 1.5))
    image = a.imshow(np.log10(table.to_numpy(float) + 1), aspect="auto", cmap="Reds", extent=[minutes[0], minutes[-1], len(table) - 0.5, -0.5],
                     interpolation="nearest")
    a.set_yticks(range(len(table)), table.index, fontsize=7)
    a.axvline(0, color="k")
    a.set(title="righe warning+ per passo (scala log10)", xlabel="minuti dall'iniezione")
    fig.colorbar(image)
    plt.show()
    late = bad[bad.t > inject].assign(template=lambda d: d.message.str.replace(r"\d+", "#", regex=True).str.slice(0, 120))
    display(late.groupby(["service", "level", "template"]).size().sort_values(ascending=False).head(12).rename("righe"))
else:
    print("Nessuna riga di warning o errore nella finestra osservata")

## Metriche, trace e log insieme

Per un servizio, sulla stessa scala di tempo: CPU dalle metriche (con il rate di wrk2 tratteggiato), p95 ed errori dalle trace, righe warning+ dai log. Una linea che si interrompe è un dato che manca, ed è anch'essa un segnale: un pod che non riparte non produce né metriche né span.

In [ ]:
service = SERVICE or (ex.target_services(run) or [run.services[0]])[0]
by_step = feats[feats.service == service].set_index("t").reindex(run.t)
service_bad = bad[bad.service == service] if len(bad) else bad
log_counts = np.histogram(service_bad.t, bins=np.r_[run.t[0] - STEP, run.t])[0] if len(service_bad) else np.zeros(len(run.t))
fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
axes[0].plot(minutes, ex.series(run, service, "cpu") * 1000)
axes[0].set_ylabel("CPU mC")
load = axes[0].twinx()
load.step(minutes, np.where(run.rate >= 0, run.rate, np.nan), where="post", color="0.6", ls="--")
load.set_ylabel("req/s wrk2")
axes[1].plot(minutes, by_step.p95_ms)
axes[1].set_ylabel("p95 ms")
axes[2].plot(minutes, by_step.error_share)
axes[2].set_ylabel("quota errori")
axes[3].bar(minutes, log_counts, width=STEP / 60)
axes[3].set_ylabel("righe warning+")
for a in axes:
    a.axvline(0, color="red", ls="--")
axes[0].set_title(f"{service}: metriche, trace e log")
axes[-1].set_xlabel("minuti dall'iniezione")
plt.show()